In [240]:
from references import zeng24_config, zeng24_question
from src.retrieval import VectorRetriever, RerankerManager, LLMHybridSummarization
from src.prompts import LLMQueryRewriter, SimplePromptConstructor
from src.llm import OpenAILLM
import os
import json
from src.utils import get_retrieval_info, get_data_chunks, get_data_chunks_by_params, get_llm_output_file
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 初始化设置以及数据库

In [241]:
cfg = zeng24_config.Zeng24fiqa()

retrieval_name, retrieval_store_path = get_retrieval_info(cfg)
print(f"Retrieval store path: {retrieval_store_path}")


# 初始化
retriever = VectorRetriever(retrieval_name=retrieval_name, 
                retrieval_store_path=retrieval_store_path, 
                retrieval_method=cfg.retrieval.method,
                embedding_provider=cfg.embedding.provider,
                embedding_model_dir=cfg.embedding.model_dir,
                data_dir_list=cfg.datastorage.raw_data_dir,
                device='cuda:1', 
                force_rebuild=False, 
                retrival_database_batch_size=256
                # top_k=cfg.retrieval.params.get("k", 15),
                # fetch_k=cfg.retrieval.params.get("fetch_k", 60),
                # score_threshold=cfg.retrieval.params.get("score_threshold", 0.75),
                # datastorage_tool=cfg.datastorage.tool,
                )

Retrieval store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Loading existing Chroma DB: ./data/fiqa
Retriever of mmr is ready.
Retriever of chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!


# 初始化LLM对话

In [242]:
# LL_Model = OpenAILLM(
#                     model = "./Models/Qwen2.5-14B-Instruct", 
#                     base_url = "http://localhost:22999/v1", 
#                     api_key = "EMPTY", 
#                     reasoning= cfg.llm.reasoning,
#                     temperature= cfg.llm.temperature,
#                     top_p= cfg.llm.top_p,
#                     max_gen_len= cfg.llm.max_gen_len)

LL_Model = OpenAILLM(
                    model = "./Models/Qwen3-14B", 
                    base_url = "http://localhost:22999/v1", 
                    api_key = "EMPTY", 
                    reasoning= cfg.llm.reasoning,
                    temperature= cfg.llm.temperature,
                    top_p= cfg.llm.top_p,
                    max_gen_len= cfg.llm.max_gen_len)

# 生成或加载问题

In [262]:
# 输入查询
queries = ["What are the causes of Volume?", "What is Profit Margin?"]

In [282]:
qrw = LLMQueryRewriter(LL_Model)

In [283]:
queries_rws = qrw.rewrite(queries, n_variants=5)

In [284]:
queries_rws

{'original_query': ['What are the causes of Volume?',
  'What is Profit Margin?'],
 'rewritten_queries': [[],
  ['What is the formula for calculating profit margin?',
   "How do you calculate profit margin and what does it indicate about a company's financial health?",
   'What are the potential drawbacks of having a high profit margin?',
   'Can you explain profit margin in simple terms for someone new to business?',
   "What factors can cause a company's profit margin to increase or decrease?"]],
 'all_queries': [['What are the causes of Volume?'],
  ['What is Profit Margin?',
   'What is the formula for calculating profit margin?',
   "How do you calculate profit margin and what does it indicate about a company's financial health?",
   'What are the potential drawbacks of having a high profit margin?',
   'Can you explain profit margin in simple terms for someone new to business?',
   "What factors can cause a company's profit margin to increase or decrease?"]]}

# 检索得到chunk

In [247]:
reranker = RerankerManager(reranker_dir=cfg.retrieval.rerank, top_n=cfg.retrieval.params.get("n", 10), device='cuda:1')

[INFO] Reranker BAAI/bge-reranker-large is ready!


In [248]:
# # 调用 retrieve 方法
# contexts, doc_ids = retriever.retrieve(queries_rws["original_query"])

# # 查看结果
# for i, q in enumerate(queries_rws["original_query"]):
#     print(f"\n🔍 Query: {q}")
#     for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
#         print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")

In [249]:
# contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# # 查看结果
# for i, q in enumerate(queries):
#     print(f"\n🔍 Query: {q}")
#     for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
#         print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")

#### 下面测试使用rewriter的格式

In [266]:
# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries_rws["all_queries"])

# 查看结果
for i, q in enumerate(queries_rws["all_queries"]):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: ['What are the causes of Volume?']
  1. [569627] Volumes are used to predict momentum of movement, not the direction of it. Large trading volumes gen...
  2. [261802] Open, high, low, close, volume. The hint is that volume on new years day is 0.  DC's comment is actu...
  3. [75680] "There are several causes of inflation. One is called cost push — that is, if the price of e.g. oil ...
  4. [446997] As far as I knew a similar law was already on the books, something about the commercial can be no lo...
  5. [79807] The daily Volume is usually compared to the average daily volume over the past 50 days for a stock. ...
  6. [51311] There are many reasons, some already covered by other answers. I have a blog post on the issue here,...
  7. [512153] Four possible reasons for the difference:...
  8. [369437] SeekingAlpha has an article about short squeezes that states: The higher the number of days to cover...
  9. [162247] There are so many unnoticed issues and concerns that arise 

In [267]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: What are the causes of Volume?
  1. [569627] Volumes are used to predict momentum of movement, not the direction of it. Large trading volumes gen...
  2. [512153] Four possible reasons for the difference:...
  3. [51311] There are many reasons, some already covered by other answers. I have a blog post on the issue here,...
  4. [79807] The daily Volume is usually compared to the average daily volume over the past 50 days for a stock. ...
  5. [369437] SeekingAlpha has an article about short squeezes that states: The higher the number of days to cover...
  6. [135877] Stock markets have not been the cause of any of the problems. They display the symptoms of the probl...
  7. [75680] "There are several causes of inflation. One is called cost push — that is, if the price of e.g. oil ...
  8. [24006] Part A solution:  Assume no turnover in A.:      Average Balance * Annual Interest - Average Balance...
  9. [103403] his is because with time you have accumulated bits and pieces th

## Summarization

In [268]:
summar = LLMHybridSummarization(LL_Model, 
                                embed_provider=cfg.embedding.provider,
                                embed_model_dir=cfg.embedding.model_dir,
                                device='cuda:1')

In [269]:
sum_chunks = summar.summarize(contexts, queries)

# 形成结构prompt

In [274]:
p_construct = SimplePromptConstructor()

In [275]:
p_construct.prefix

['context: ', 'question: ', 'answer:']

In [276]:
end_ppt_contexts = p_construct.batch_construct(queries, contexts)
end_ppt_sum = p_construct.batch_construct(queries, sum_chunks)
end_ppt_query = p_construct.batch_construct(queries, [])

In [277]:
end_ppt_contexts

['context: Volumes are used to predict momentum of movement, not the direction of it. Large trading volumes generally tend to create a price breakout in either positive or negative direction. Especially in relatively illiquid stocks (like small caps), sudden volume surges can create sharp price fluctuations.\n\nFour possible reasons for the difference:\n\nThere are many reasons, some already covered by other answers. I have a blog post on the issue here, and I\'ll summarize:\n\nThe daily Volume is usually compared to the average daily volume over the past 50 days for a stock.  High volume is usually considered to be 2 or more times the average daily volume over the last 50 days for that stock, however some traders might set the crireia to be 3x or 4x the ADV for confirmation of a particular pattern or event. The volume is compared to the ADV of the stock itself, as comparing it to the volume of other stocks would be like comparing apples with oranges, as difference companies would have

In [278]:
end_ppt_sum

['context: The sentence does not provide information on the causes of Volume.\n\n4\n\nThe sentence does not provide specific causes of Volume. No numerical information is present.\n\nThe daily Volume is compared to the average daily volume (ADV) over the past 50 days. High volume is usually 2 or more times the ADV, with some traders using 3x or 4x the ADV for confirmation. Comparing to other stocks is invalid due to differences in total stocks, liquidity, and volatility.\n\nThe current short interest (numerator) increases. The average daily volume (denominator) decreases.\n\nThe problems have been caused by the corruption of government (lobbying and political donations), insufficient, excessive and inappropriate regulation, and ignorance on the part of those who vote (demanding more services and less taxes and worrying more about wedge issues than sound governance).\n\nThe sentence does not mention "Volume" or its causes. It discusses causes of inflation: cost push (e.g., oil price inc

# 输入LLM进行测试

In [279]:
LL_Model.infer("who are you?")

("Hello! I'm Qwen, a large language model developed by Alibaba Cloud. I can answer questions, create text, and have conversations on a wide range of topics. I'm also multilingual and can assist with various tasks. How can I help you today? 😊",
 'Okay, the user asked, "who are you?" I need to respond clearly. First, I should introduce myself as Qwen, a large language model developed by Alibaba Cloud. I should mention my capabilities, like answering questions, creating text, and having conversations. Also, I need to highlight my multilingual support and the ability to handle various tasks. But I should keep it concise and friendly. Let me make sure I don\'t use any markdown and keep the response natural.')

In [285]:
LL_Model.batch_infer(end_ppt_sum)

(['The provided context does not explicitly mention the causes of **Volume** (likely referring to trading volume in a financial context). While the text discusses factors like **short interest**, **average daily volume (ADV)**, and formulas involving **Volume**, it does not identify specific reasons or drivers for changes in Volume. \n\nKey points from the context:  \n1. **Volume** is compared to the **average daily volume (ADV)** over 50 days, but no causes for fluctuations in Volume are stated.  \n2. **Short interest** (numerator) increasing and **ADV** (denominator) decreasing are noted, but this is a relationship between metrics, not a cause of Volume itself.  \n3. Other sections discuss unrelated topics (e.g., inflation, formulas, sentimental value), none of which address the causes of Volume.  \n\n**Conclusion**: The context does not provide specific causes for **Volume**. If referring to general financial principles, Volume is typically influenced by factors like **market sentim

In [286]:
LL_Model.batch_infer(end_ppt_contexts)

(["The causes of trading volume in financial markets, particularly in stocks, are multifaceted and influenced by both internal and external factors. Here's a structured breakdown based on the context and broader market dynamics:\n\n### 1. **Market Events and News**  \n   - **Earnings Reports**: Surprises in quarterly earnings can trigger significant trading activity as investors adjust positions.  \n   - **Economic Data**: Releases like GDP, employment figures, or interest rate decisions can drive volume as traders react to macroeconomic trends.  \n   - **Company-Specific News**: Mergers, acquisitions, product launches, or regulatory changes often lead to increased trading.  \n\n### 2. **Short Interest and Short Squeezes**  \n   - **High Short Interest**: A large number of short sellers (borrowing shares to bet on price declines) increases the potential for a short squeeze. If short sellers are forced to cover (buy back shares), it can create sudden demand, spiking volume.  \n   - **Lo

In [ ]:
LL_Model.batch_infer(queries)

In [232]:
answers, reasons = LL_Model.batch_infer(end_ppt_contexts)

In [235]:
# 保存结果
output_dir = cfg.expconfig.output_dir
os.makedirs(output_dir, exist_ok=True)

answers_path = os.path.join(output_dir, get_llm_output_file(cfg))
with open(answers_path, "w", encoding="utf-8") as f_a:
    json.dump(answers, f_a, ensure_ascii=False, indent=2)

reasons_path = answers_path.replace(".json", "_reasoning.json")
with open(reasons_path, "w", encoding="utf-8") as f_r:
    json.dump(reasons, f_r, ensure_ascii=False, indent=2)

In [236]:
reasons_path

'./exp/fiqa/vector-chroma/bge-large-en-v1_5-Qwen2_5-14B-Instruct/mmr-15-BAAI/bge-reranker-large/outputs-Qwen2.5-14B-Instruct-0-4096-4096_reasoning.json'